In [ ]:
import psutil
import torch
import polars as pl
import json
import os
from datasets import load_dataset
from collections import defaultdict
import gc
import numpy as np

os.environ["HF_TOKEN"] = "hugging_face_token" # Подставить hugging face токен

## Проверка ресурсов

In [3]:
ram_gb = psutil.virtual_memory().available / (1024**3)
print(f"✅ Доступно RAM: {ram_gb:.2f} ГБ")
print(f"✅ GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Модель GPU: {torch.cuda.get_device_name(0)}")

✅ Доступно RAM: 29.61 ГБ
✅ GPU доступен: True
   Модель GPU: Tesla T4


In [4]:
MIN_INTERACTIONS = 10
MAX_INTERACTIONS = 150
MAX_USERS_TO_KEEP = 50000

print(f"⚙️ Фильтр: от {MIN_INTERACTIONS} до {MAX_INTERACTIONS} взаимодействий на пользователя")

⚙️ Фильтр: от 10 до 150 взаимодействий на пользователя


In [5]:
import socket
try:
    socket.gethostbyname('huggingface.co')
    print("DNS работает")
except Exception as e:
    print(f"Проблема с DNS: {e}")

DNS работает


In [6]:
dataset = load_dataset("mgor/EDNet", "kt1", streaming=True, split="train")

user_interactions = defaultdict(list)
valid_users_count = 0

for i, row in enumerate(dataset):
    user_id = row["subject_id"] 
    item_id = row["question_id"]
    timestamp = row["timestamp"]
    
    if len(user_interactions[user_id]) >= MAX_INTERACTIONS:
        del user_interactions[user_id]
        continue
        
    user_interactions[user_id].append({
        "timestamp": timestamp,
        "item_id": item_id
    })
    
    if i > 0 and i % 10_000_000 == 0:
        print(f"   Обработано {i:,} строк. Текущих активных пользователей в памяти: {len(user_interactions)}")
        
    # Для отладки
    # if i > 5_000_000:
    #     print("Тестовый лимит строк достигнут.")
    #     break

README.md: 0.00B [00:00, ?B/s]

   Обработано 10,000,000 строк. Текущих активных пользователей в памяти: 83317
   Обработано 20,000,000 строк. Текущих активных пользователей в памяти: 166198
   Обработано 30,000,000 строк. Текущих активных пользователей в памяти: 248066
   Обработано 40,000,000 строк. Текущих активных пользователей в памяти: 331571
   Обработано 50,000,000 строк. Текущих активных пользователей в памяти: 412080
   Обработано 60,000,000 строк. Текущих активных пользователей в памяти: 493518
   Обработано 70,000,000 строк. Текущих активных пользователей в памяти: 575402
   Обработано 80,000,000 строк. Текущих активных пользователей в памяти: 656705
   Обработано 90,000,000 строк. Текущих активных пользователей в памяти: 739798


## Фильтрация по минимальному количеству взаимодействий

In [7]:
final_sequences = []

for user_id, interactions in user_interactions.items():
    if len(interactions) >= MIN_INTERACTIONS:
        # Сортировка по времени
        interactions.sort(key=lambda x: x["timestamp"])
        final_sequences.append({
            "user_id": user_id,
            "sequence": [x["item_id"] for x in interactions],
            "length": len(interactions)
        })
        
    if len(final_sequences) >= MAX_USERS_TO_KEEP:
        break

del user_interactions
gc.collect()

131

In [8]:
print(f"Сформировано {len(final_sequences)} валидных последовательностей.")

Сформировано 50000 валидных последовательностей.


## Базовая статистика

In [9]:
lengths = [seq["length"] for seq in final_sequences]
print(f"   Средняя длина: {np.mean(lengths):.1f}")
print(f"   Медиана: {np.median(lengths):.1f}")
print(f"   Максимальная: {np.max(lengths)}")

   Средняя длина: 43.9
   Медиана: 30.0
   Максимальная: 150


## Формирование словаря

In [10]:
all_items = set()
for seq in final_sequences:
    all_items.update(seq["sequence"])

# 0: [PAD] (для выравнивания батчей)
# 1: [MASK] (для BERT4Rec / gSASRec)
item_list = ["[PAD]", "[MASK]"] + sorted(list(all_items))
item2idx = {item: idx for idx, item in enumerate(item_list)}
idx2item = {idx: item for item, idx in item2idx.items()}

print(f"Размер словаря: {len(item2idx)}")

Размер словаря: 12081


## Создание User Vocab

In [11]:
user_list = [seq["user_id"] for seq in final_sequences]
user2idx = {user: idx for idx, user in enumerate(user_list)}

## Преобразование последовательностей в индексы

In [12]:
processed_sequences = []
for seq in final_sequences:
    processed_sequences.append({
        "user_idx": user2idx[seq["user_id"]],
        "item_indices": [item2idx[item] for item in seq["sequence"]],
        "length": seq["length"]
    })

print("Последовательности закодированы в индексы.")

Последовательности закодированы в индексы.


## Разделение данных на подвыборки

In [13]:
train_data = []
val_data = []
test_data = []

for seq in processed_sequences:
    items = seq["item_indices"]
    user = seq["user_idx"]
    
    if len(items) >= 3:
        train_data.append({"user_idx": user, "sequence": items[:-2]})
        val_data.append({"user_idx": user, "sequence": items[:-1], "target": items[-2]})
        test_data.append({"user_idx": user, "sequence": items, "target": items[-1]})
    else:
        train_data.append({"user_idx": user, "sequence": items})

print(f"   Train: {len(train_data)}")
print(f"   Val:   {len(val_data)}")
print(f"   Test:  {len(test_data)}")

   Train: 50000
   Val:   50000
   Test:  50000


In [14]:
import pickle

OUTPUT_DIR = "./ednet_prepared"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Сохраняем словари 
with open(os.path.join(OUTPUT_DIR, "item2idx.json"), "w") as f:
    json.dump(item2idx, f)
with open(os.path.join(OUTPUT_DIR, "idx2item.json"), "w") as f:
    json.dump(idx2item, f)

# 2. Сохраняем данные в формате Parquet
pl.DataFrame(train_data).write_parquet(os.path.join(OUTPUT_DIR, "train.parquet"))
pl.DataFrame(val_data).write_parquet(os.path.join(OUTPUT_DIR, "val.parquet"))
pl.DataFrame(test_data).write_parquet(os.path.join(OUTPUT_DIR, "test.parquet"))

# 3. Сохраним пару примеров для теста
sample_test = test_data[:5]
with open(os.path.join(OUTPUT_DIR, "sample_inference.json"), "w") as f:
    json.dump(sample_test, f, indent=2)

print(f"Данные успешно сохранены в папку: `{OUTPUT_DIR}`")

Данные успешно сохранены в папку: `./ednet_prepared`
